# Introdução à Engenharia de Prompt
Engenharia de prompt é o processo de projetar e otimizar prompts para tarefas de processamento de linguagem natural. Isso envolve selecionar os prompts certos, ajustar seus parâmetros e avaliar seu desempenho. A engenharia de prompt é crucial para alcançar alta precisão e eficiência em modelos de PLN. Nesta seção, exploraremos os fundamentos da engenharia de prompt usando os modelos OpenAI para exploração.


### Exercício 1: Tokenização
Explore a Tokenização usando tiktoken, um tokenizer rápido de código aberto da OpenAI
Veja [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) para mais exemplos.


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### Exercício 2: Validar Configuração da Chave dos Modelos Microsoft Foundry

> **Nota:** O GitHub Models será descontinuado no final de julho de 2026. Este exercício usa [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) em seu lugar, que oferece o mesmo catálogo de modelos para teste gratuito e a experiência do Azure AI Inference SDK.

Execute o código abaixo para verificar se o endpoint dos seus Modelos Microsoft Foundry está configurado corretamente. O código apenas tenta um prompt simples e básico e valida a conclusão. A entrada `oh say can you see` deve completar algo como `by the dawn's early light..`


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

# temperature/top_p need a non-reasoning model (gpt-5 rejects them), so use a Llama model here.
model_name = os.environ.get("AZURE_INFERENCE_CHAT_MODEL", "Llama-3.3-70B-Instruct")

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

def get_completion(prompt, client, model_name, temperature=1.0, max_tokens=1000, top_p=1.0):
    response = client.complete(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        model=model_name,
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p
    )
    return response.choices[0].message.content

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt, client, model_name)
print(response)


That line is the opening lyric of "The Star-Spangled Banner," the national anthem of the United States, written by Francis Scott Key. If you'd like more information or analysis, feel free to ask!


### Exercício 3: Fabricações
Explore o que acontece quando você pede ao LLM para retornar completions para um prompt sobre um tópico que pode não existir, ou sobre tópicos que ele pode não conhecer porque estavam fora do conjunto de dados pré-treinados (mais recentes). Veja como a resposta muda se você tentar um prompt diferente, ou um modelo diferente.


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt, client, model_name)
print(response)

### Exercício 4: Baseado em Instrução 
Use a variável "text" para definir o conteúdo principal 
e a variável "prompt" para fornecer uma instrução relacionada a esse conteúdo principal.

Aqui pedimos ao modelo para resumir o texto para um aluno da segunda série


In [4]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt, client, model_name)
print(response)

Jupiter is the fifth planet from the Sun and the biggest one in our Solar System. It's made of gas and is much bigger than all the other planets put together! You can see Jupiter in the night sky because it's very bright. People have noticed it for a really long time and named it after a Roman god.


### Exercício 5: Prompt Complexo 
Tente uma solicitação que tenha mensagens do sistema, do usuário e do assistente 
O sistema define o contexto do assistente
Mensagens do usuário e do assistente fornecem contexto para uma conversa de múltiplas rodadas

Note como a personalidade do assistente é definida como "sarcástica" no contexto do sistema. 
Tente usar um contexto de personalidade diferente. Ou tente uma série diferente de mensagens de entrada/saída


In [5]:
response = client.complete(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ]
)
print(response.choices[0].message.content)

Oh, you mean the famous 2020 World Series that wasn’t in a regular location? That was the year they played in the glamorous Arlington, Texas, at Globe Life Field.


### Exercício: Explore Sua Intuição
Os exemplos acima mostram padrões que você pode usar para criar novos prompts (simples, complexos, instruções etc.) - tente criar outros exercícios para explorar algumas das outras ideias sobre as quais falamos, como exemplos, dicas e mais.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido usando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, por favor, esteja ciente de que traduções automatizadas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
